# Chapter 02. OpenAI Function Calling으로 도구 체계 구축하기

## 학습 목표
- Function Calling의 원리를 이해한다
- JSON Schema로 도구를 정의하는 방법을 학습한다
- 8개 도구를 OpenAI Function Calling으로 정의한다
- 도구 실행 결과를 메시지에 주입하는 패턴을 이해한다

## 환경 설정

In [1]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. Function Calling 기본 원리

In [2]:
# ============================================================
# Function Calling 기본 예제
# ============================================================
# 간단한 데이터셋 분석 도구를 정의하고 LLM이 선택하게 한다.

import json
import pandas as pd
import numpy as np

# 도구 1: 데이터셋 분석
def analyze_dataset(dataset_path: str, analysis_type: str = "basic") -> dict:
    """CSV 데이터셋을 분석하는 실제 함수이다."""
    df = pd.read_csv(dataset_path)
    result = {
        "num_rows": len(df),
        "num_columns": len(df.columns),
        "columns": list(df.columns),
        "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
    }
    if analysis_type == "detailed":
        result["describe"] = df.describe().to_dict()
    return result

# 도구 2: 평균 차이 계산
def calculate_mean_difference(dataset_path: str, treatment_col: str, outcome_col: str) -> dict:
    """처치군과 대조군의 평균 차이를 계산하는 함수이다."""
    df = pd.read_csv(dataset_path)
    treated = df[df[treatment_col] == 1][outcome_col]
    control = df[df[treatment_col] == 0][outcome_col]
    return {
        "mean_treated": float(treated.mean()),
        "mean_control": float(control.mean()),
        "difference": float(treated.mean() - control.mean()),
        "n_treated": int(len(treated)),
        "n_control": int(len(control))
    }

In [4]:
# JSON Schema로 도구 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "analyze_dataset",
            "description": "CSV 데이터셋의 기본 정보(행수, 열수, 컬럼, 타입)를 분석한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_path": {"type": "string", "description": "CSV 파일 경로"},
                    "analysis_type": {"type": "string", "enum": ["basic", "detailed"], "description": "분석 수준"}
                },
                "required": ["dataset_path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_mean_difference",
            "description": "처치군과 대조군의 결과 변수 평균 차이를 계산한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_path": {"type": "string", "description": "CSV 파일 경로"},
                    "treatment_col": {"type": "string", "description": "처치 변수 컬럼명"},
                    "outcome_col": {"type": "string", "description": "결과 변수 컬럼명"}
                },
                "required": ["dataset_path", "treatment_col", "outcome_col"]
            }
        }
    }
]

In [5]:
# 도구 레지스트리 (이름 → 함수 매핑)
tool_registry = {
    "analyze_dataset": analyze_dataset,
    "calculate_mean_difference": calculate_mean_difference
}

print("도구 2개 정의 완료!")
print("- analyze_dataset: 데이터셋 기본 분석")
print("- calculate_mean_difference: 평균 차이 계산")

도구 2개 정의 완료!
- analyze_dataset: 데이터셋 기본 분석
- calculate_mean_difference: 평균 차이 계산


In [6]:
# ============================================================
# LLM에게 질문하여 도구를 선택하게 한다
# ============================================================

messages = [
    {"role": "system", "content": "너는 인과추론 분석 에이전트이다. 사용자의 질문에 적절한 도구를 사용하여 분석한다."},
    {"role": "user", "content": "data/all_data/online_classroom.csv 데이터를 먼저 분석해줘."}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

# 응답 분석
message = response.choices[0].message
print("=== LLM 응답 분석 ===")
print(f"finish_reason: {response.choices[0].finish_reason}")
print(f"도구 호출 여부: {'있음' if message.tool_calls else '없음'}")

if message.tool_calls:
    for tc in message.tool_calls:
        print(f"\n선택된 도구: {tc.function.name}")
        print(f"인자: {json.dumps(json.loads(tc.function.arguments), indent=2, ensure_ascii=False)}")

=== LLM 응답 분석 ===
finish_reason: tool_calls
도구 호출 여부: 있음

선택된 도구: analyze_dataset
인자: {
  "dataset_path": "data/all_data/online_classroom.csv",
  "analysis_type": "basic"
}


## 2. 도구 실행 결과 주입 패턴

In [7]:
# ============================================================
# 완전한 Function Calling 루프
# ============================================================
# 1단계: 사용자 질문 → LLM이 도구 선택
# 2단계: 도구 실행
# 3단계: 결과를 tool 메시지로 주입
# 4단계: LLM이 최종 답변 생성

DATA_PATH = "./dataset/online_classroom.csv"

messages = [
    {"role": "system", "content": "너는 인과추론 분석 에이전트이다. 도구를 활용하여 데이터를 분석하고 인과효과를 추정한다."},
    {"role": "user", "content": f"{DATA_PATH} 데이터에서 온라인 수업(format_ol)이 시험 점수(falsexam)에 미치는 효과를 분석해줘. 먼저 데이터를 분석한 후 평균 차이를 계산해줘."}
]

# 최대 3번의 도구 호출 루프
for step in range(3):
    print(f"\n{'='*50}")
    print(f"Step {step + 1}: LLM 호출")
    print(f"{'='*50}")
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    assistant_message = response.choices[0].message
    
    # 도구 호출이 없으면 최종 답변
    if not assistant_message.tool_calls:
        print("[최종 답변]")
        print(assistant_message.content)
        break
    
    # assistant 메시지를 대화에 추가 (tool_calls 포함)
    messages.append(assistant_message)
    
    # 각 도구 호출을 실행하고 결과를 주입한다
    for tc in assistant_message.tool_calls:
        func_name = tc.function.name
        func_args = json.loads(tc.function.arguments)
        
        print(f"\n  [도구 실행] {func_name}")
        print(f"  인자: {func_args}")
        
        # 실제 함수 실행
        result = tool_registry[func_name](**func_args)
        print(f"  결과: {json.dumps(result, indent=2, ensure_ascii=False)[:200]}...")
        
        # 결과를 tool 메시지로 주입
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,  # 반드시 tool_call_id를 매칭해야 한다
            "content": json.dumps(result, ensure_ascii=False)
        })


Step 1: LLM 호출

  [도구 실행] analyze_dataset
  인자: {'dataset_path': '../data/all_data/online_classroom.csv', 'analysis_type': 'basic'}
  결과: {
  "num_rows": 323,
  "num_columns": 10,
  "columns": [
    "gender",
    "asian",
    "black",
    "hawaiian",
    "hispanic",
    "unknown",
    "white",
    "format_ol",
    "format_blended",
    ...

Step 2: LLM 호출

  [도구 실행] calculate_mean_difference
  인자: {'dataset_path': '../data/all_data/online_classroom.csv', 'treatment_col': 'format_ol', 'outcome_col': 'falsexam'}
  결과: {
  "mean_treated": 73.63526308510637,
  "mean_control": 77.85552344978166,
  "difference": -4.220260364675283,
  "n_treated": 94,
  "n_control": 229
}...

Step 3: LLM 호출
[최종 답변]
데이터 분석 결과, `online_classroom.csv` 파일에는 총 323개의 행과 10개의 열이 있습니다. 각 열의 이름과 데이터 타입은 다음과 같습니다:

- **gender**: int64
- **asian**: float64
- **black**: float64
- **hawaiian**: float64
- **hispanic**: float64
- **unknown**: float64
- **white**: float64
- **format_ol**: int64 (온라인 수업 여부)
- **format_blende

## 3. 8개 도구를 JSON Schema로 정의

In [10]:
# ============================================================
# 8개 도구를 OpenAI Function Calling용 JSON Schema로 정의한다
# ============================================================


cais_tools = [
    {
        "type": "function",
        "function": {
            "name": "input_parser",
            "description": "사용자의 자연어 질문에서 인과 질문, 데이터셋 경로, 설명을 추출한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "사용자의 인과 질문"},
                    "dataset_path": {"type": "string", "description": "데이터셋 파일 경로"},
                    "dataset_description": {"type": "string", "description": "데이터셋 설명 (선택)"}
                },
                "required": ["query", "dataset_path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "dataset_analyzer",
            "description": "데이터셋을 로드하고 행수, 열수, 변수 타입, 시간 구조, 패널 데이터 여부 등을 분석한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_path": {"type": "string", "description": "CSV 파일 경로"},
                    "dataset_description": {"type": "string", "description": "데이터셋 설명"},
                    "original_query": {"type": "string", "description": "원래 질문"}
                },
                "required": ["dataset_path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "query_interpreter",
            "description": "LLM을 활용하여 질문에서 처치변수, 결과변수, 공변량, 도구변수, RCT 여부를 식별한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query_text": {"type": "string", "description": "인과 질문"},
                    "columns": {"type": "array", "items": {"type": "string"}, "description": "데이터셋 컬럼 목록"},
                    "dataset_description": {"type": "string", "description": "데이터셋 설명"}
                },
                "required": ["query_text", "columns"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "method_selector",
            "description": "의사결정 트리를 기반으로 최적의 인과추론 방법론(DiM, OLS, DiD, IV, RDD, PSM, PSW 등)을 선택한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "treatment_variable": {"type": "string"},
                    "outcome_variable": {"type": "string"},
                    "is_rct": {"type": "boolean", "description": "무작위 실험 여부"},
                    "has_temporal_structure": {"type": "boolean"},
                    "has_instrument": {"type": "boolean"},
                    "has_running_variable": {"type": "boolean"},
                    "treatment_type": {"type": "string", "enum": ["binary", "continuous", "categorical"]}
                },
                "required": ["treatment_variable", "outcome_variable", "is_rct"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "method_validator",
            "description": "선택된 방법론의 핵심 가정(평행추세, 배제제약, 비혼동성 등)을 데이터로 검증한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "method": {"type": "string", "description": "선택된 방법론"},
                    "dataset_path": {"type": "string"},
                    "treatment_variable": {"type": "string"},
                    "outcome_variable": {"type": "string"},
                    "covariates": {"type": "array", "items": {"type": "string"}}
                },
                "required": ["method", "dataset_path", "treatment_variable", "outcome_variable"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "method_executor",
            "description": "검증된 방법론으로 인과효과를 추정하고 통계량(ATE, SE, p-value, CI)을 반환한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "method": {"type": "string", "enum": ["diff_in_means", "linear_regression", "difference_in_differences", "instrumental_variable", "regression_discontinuity_design", "propensity_score_matching", "propensity_score_weighting", "backdoor_adjustment"]},
                    "dataset_path": {"type": "string"},
                    "treatment_variable": {"type": "string"},
                    "outcome_variable": {"type": "string"},
                    "covariates": {"type": "array", "items": {"type": "string"}}
                },
                "required": ["method", "dataset_path", "treatment_variable", "outcome_variable"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "explanation_generator",
            "description": "인과 분석 결과를 비전공자도 이해할 수 있는 자연어 해석으로 변환한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "method_used": {"type": "string"},
                    "effect_estimate": {"type": "number"},
                    "p_value": {"type": "number"},
                    "confidence_interval": {"type": "array", "items": {"type": "number"}},
                    "original_query": {"type": "string"}
                },
                "required": ["method_used", "effect_estimate", "original_query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "output_formatter",
            "description": "분석 결과, 해석, 방법론 정보를 구조화된 최종 보고서 형태로 포맷한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "results": {"type": "object", "description": "분석 결과 딕셔너리"},
                    "explanation": {"type": "string", "description": "결과 해석 텍스트"},
                    "method_info": {"type": "object", "description": "방법론 정보"}
                },
                "required": ["results", "explanation"]
            }
        }
    }
]

In [11]:
# 도구 목록 요약 출력
print("=== 8개 도구 (OpenAI Function Calling 형태) ===")
for t in cais_tools:
    func = t["function"]
    params = func["parameters"]["properties"]
    required = func["parameters"].get("required", [])
    print(f"\n{func['name']}:")
    print(f"  설명: {func['description'][:60]}...")
    print(f"  파라미터: {list(params.keys())}")
    print(f"  필수: {required}")

=== 8개 도구 (OpenAI Function Calling 형태) ===

input_parser:
  설명: 사용자의 자연어 질문에서 인과 질문, 데이터셋 경로, 설명을 추출한다....
  파라미터: ['query', 'dataset_path', 'dataset_description']
  필수: ['query', 'dataset_path']

dataset_analyzer:
  설명: 데이터셋을 로드하고 행수, 열수, 변수 타입, 시간 구조, 패널 데이터 여부 등을 분석한다....
  파라미터: ['dataset_path', 'dataset_description', 'original_query']
  필수: ['dataset_path']

query_interpreter:
  설명: LLM을 활용하여 질문에서 처치변수, 결과변수, 공변량, 도구변수, RCT 여부를 식별한다....
  파라미터: ['query_text', 'columns', 'dataset_description']
  필수: ['query_text', 'columns']

method_selector:
  설명: 의사결정 트리를 기반으로 최적의 인과추론 방법론(DiM, OLS, DiD, IV, RDD, PSM, PSW ...
  파라미터: ['treatment_variable', 'outcome_variable', 'is_rct', 'has_temporal_structure', 'has_instrument', 'has_running_variable', 'treatment_type']
  필수: ['treatment_variable', 'outcome_variable', 'is_rct']

method_validator:
  설명: 선택된 방법론의 핵심 가정(평행추세, 배제제약, 비혼동성 등)을 데이터로 검증한다....
  파라미터: ['method', 'dataset_path', 'treatment_variable', 'outcome_variable', 'covariates']
  필수: